# Use Case 3 - ETL for ML Preparation on AWS

This lab treats churn modeling as a **data engineering process** first and a machine-learning workflow second.


## What this lab covers
- Amazon S3 raw and curated zones
- Notebook ETL profiling and transformation
- Optional AWS Glue productionization bridge
- Amazon SageMaker training and Model Registry


## ETL flow in one line
`Extract from S3 raw -> Transform and validate in notebook -> Load curated train/test to S3 -> Train in SageMaker -> Register model version`


In [ ]:
import json, pandas as pd, numpy as np
from pathlib import Path

BASE = Path("..")
raw_path = BASE / "04_Datasets" / "raw" / "telco_customer_churn_sample.csv"
train_out = BASE / "04_Datasets" / "ml_ready" / "train.csv"
test_out = BASE / "04_Datasets" / "ml_ready" / "test.csv"
print(raw_path)


## 1. Extract - land raw data
In the live AWS demo, show the same file in `s3://<bucket>/churn/raw/` before reading it here locally.


In [ ]:
df_raw = pd.read_csv(raw_path)
df_raw.head()


## 2. Profile the source before transformation
Profile nulls, blanks, duplicates, and label quality. This is still ETL. The downstream consumer just happens to be a model.


In [ ]:
profile = pd.DataFrame({"dtype": df_raw.dtypes.astype(str), "null_count": df_raw.isna().sum(), "blank_count": df_raw.astype(str).apply(lambda s: s.str.strip().eq("")).sum()}).sort_values(["null_count","blank_count"], ascending=False)
profile.head(15)


In [ ]:
print("Duplicate customer IDs:", df_raw["customerID"].duplicated().sum())
print(df_raw["Churn"].value_counts(dropna=False))


## 3. Transform - standardize fields and engineer features


In [ ]:
from sklearn.model_selection import train_test_split

df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", np.nan), errors="coerce")
df["label"] = df["Churn"].map({"Yes":1, "No":0})
df["is_new_customer"] = (df["tenure"] <= 6).astype(int)
df["monthly_charge_band"] = pd.cut(df["MonthlyCharges"], bins=[0,35,70,200], labels=["Low","Medium","High"], include_lowest=True)
df["avg_monthly_spend_gap"] = df["TotalCharges"] - (df["tenure"] * df["MonthlyCharges"])
feature_df = df.copy()
feature_df.head()


## 4. Validate curated data before publish


In [ ]:
validation_summary = {
    "row_count": int(feature_df.shape[0]),
    "null_totalcharges": int(feature_df["TotalCharges"].isna().sum()),
    "null_labels": int(feature_df["label"].isna().sum()),
    "duplicate_customer_ids": int(feature_df["customerID"].duplicated().sum())
}
validation_summary


## 5. Load - publish train/test outputs


In [ ]:
train_df, test_df = train_test_split(feature_df, test_size=0.2, random_state=42, stratify=feature_df["label"])
train_df.to_csv(train_out, index=False)
test_df.to_csv(test_out, index=False)
pd.DataFrame([validation_summary]).to_csv(BASE / "05_Artifacts" / "validation_summary_generated.csv", index=False)
print(train_df.shape, test_df.shape)


## 6. Optional AWS Glue bridge
Open `06_Assets/code/optional_glue_job_uc3.py` to explain how the notebook transforms can become a repeatable Glue PySpark job.


## 7. Train and register
In the live demo, use SageMaker to train and then explain Model Registry as the governed publish point for the model artifact.
